# Module 07: Preprocessing with fMRIPrep (Singularity/Apptainer)

This notebook covers running fMRIPrep on HPC clusters using Singularity or Apptainer.

**When to use Singularity instead of Docker:**
- HPC clusters (Slurm, PBS) that do not allow Docker
- Environments where you don't have root/sudo access
- Apptainer is the successor to Singularity (same syntax)

## 1. Pulling the fMRIPrep Singularity Image

```bash
# On your HPC login node (may take 20-30 minutes)
singularity pull docker://nipreps/fmriprep:23.1.4
# Produces: fmriprep_23.1.4.sif

# Or use Apptainer
apptainer pull docker://nipreps/fmriprep:23.1.4
```

Store the `.sif` file in a persistent location (e.g., `/scratch/images/`).

## 2. Running fMRIPrep with Singularity

```bash
singularity run --cleanenv \
  -B /path/to/bids:/data:ro \
  -B /path/to/output:/out \
  -B /path/to/license.txt:/opt/freesurfer/license.txt:ro \
  -B /path/to/work:/work \
  /path/to/fmriprep_23.1.4.sif \
  /data /out participant \
  --participant-label 01 \
  --work-dir /work \
  --output-spaces MNI152NLin2009cAsym:res-2 T1w \
  --dummy-scans 4 \
  --skip-bids-validation
```

**Key differences from Docker:**
- Use `-B` instead of `-v` for volume mounts
- Use `--cleanenv` to avoid environment variable conflicts
- Need to set `TEMPLATEFLOW_HOME` manually

## 3. SLURM Job Script Template

```bash
#!/usr/bin/env bash
#SBATCH --job-name=fmriprep
#SBATCH --time=08:00:00
#SBATCH --mem=16G
#SBATCH --cpus-per-task=8
#SBATCH --output=logs/fmriprep_%j.out

SUBJECT=${1:-01}
BIDS_DIR=/project/data/bids
OUT_DIR=/project/data/fmriprep
SIF=/scratch/images/fmriprep_23.1.4.sif
FS_LICENSE=/project/freesurfer_license.txt

export TEMPLATEFLOW_HOME=/scratch/templateflow
mkdir -p $TEMPLATEFLOW_HOME $OUT_DIR/work

singularity run --cleanenv \
  -B $BIDS_DIR:/data:ro \
  -B $OUT_DIR:/out \
  -B $FS_LICENSE:/opt/freesurfer/license.txt:ro \
  -B $OUT_DIR/work:/work \
  -B $TEMPLATEFLOW_HOME:/templateflow \
  --env TEMPLATEFLOW_HOME=/templateflow \
  $SIF /data /out participant \
  --participant-label $SUBJECT \
  --work-dir /work \
  --output-spaces MNI152NLin2009cAsym:res-2 T1w \
  --dummy-scans 4 --fd-spike-threshold 0.5 \
  --nthreads 8 --mem-mb 15000 \
  --skip-bids-validation
```

Submit as a job array:
```bash
sbatch --array=01-20 run_fmriprep_slurm.sh
```

## 4. TemplateFlow Setup

TemplateFlow is used by fMRIPrep to download brain templates. On HPC you may need to:

```bash
# Pre-download templates on login node
export TEMPLATEFLOW_HOME=/scratch/templateflow
python -c "from templateflow import api; api.get('MNI152NLin2009cAsym')"
```

Then bind the cache directory in your Singularity command.

## Summary

Key differences between Docker and Singularity:

| Feature | Docker | Singularity |
|---------|--------|-------------|
| Root needed | Yes (daemon) | No |
| HPC compatible | Rarely | Yes |
| Volume flag | `-v` | `-B` |
| Clean env | Automatic | `--cleanenv` |
| Image format | layers | `.sif` file |

**Next:** Module 08 — Building Custom Nipype Workflows